Importação das bibliotecas.

In [1]:
import cv2
import numpy as np
import os
import time


## Cadastro de pessoas e criação da base de dados

In [2]:



nome = input("Digite o nome da pessoa para cadastro: ")
pasta = f"dataset/{nome}"
os.makedirs(pasta, exist_ok=True)

detector = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

cap = cv2.VideoCapture(0)
contador = 0

while contador < 150:
    ret, frame = cap.read()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = detector.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        rosto = gray[y:y+h, x:x+w]
        rosto = cv2.resize(rosto, (100, 100))
        contador += 1
        cv2.imwrite(f"{pasta}/{contador}.jpg", rosto)
        cv2.rectangle(frame, (x,y), (x+w,y+h), (0,255,0), 2)

    cv2.imshow("Cadastro Facial", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


## Treinamento do modelo LBPH

In [3]:



recognizer = cv2.face.LBPHFaceRecognizer_create()
faces = []
labels = []
label_map = {}
label_id = 0

for pessoa in os.listdir("dataset"):
    pasta_pessoa = os.path.join("dataset", pessoa)
    if not os.path.isdir(pasta_pessoa):
        continue

    label_map[label_id] = pessoa

    for img_nome in os.listdir(pasta_pessoa):
        img = cv2.imread(os.path.join(pasta_pessoa, img_nome), cv2.IMREAD_GRAYSCALE)
        faces.append(img)
        labels.append(label_id)

    label_id += 1

recognizer.train(np.array(faces), np.array(labels))

os.makedirs("model", exist_ok=True)
recognizer.save("model/lbph_model.yml")
np.save("model/labels.npy", label_map)

print("Modelo treinado com sucesso")


Modelo treinado com sucesso



## Liveness

In [4]:
recognizer = cv2.face.LBPHFaceRecognizer_create()
recognizer.read("model/lbph_model.yml")
label_map = np.load("model/labels.npy", allow_pickle=True).item()

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)
eye_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_eye.xml"
)

cap = cv2.VideoCapture(0)

prev_face = None
motion_history = []
eye_history = []

WINDOW = 20
MOTION_THRESHOLD = 2.5
STATIC_LIMIT = 8
LIVENESS_HOLD_TIME = 2.5  # segundos

liveness_until = 0

while True:
    ret, frame = cap.read()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    now = time.time()

    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        face_gray = gray[y:y+h, x:x+w]
        face_gray = cv2.resize(face_gray, (100,100))

        eyes = eye_cascade.detectMultiScale(face_gray, 1.1, 5)

        # Micro movimento
        motion = 0
        if prev_face is not None:
            diff = cv2.absdiff(face_gray, prev_face)
            motion = np.mean(diff)

        prev_face = face_gray.copy()

        motion_history.append(motion)
        eye_history.append(len(eyes))

        motion_history[:] = motion_history[-WINDOW:]
        eye_history[:] = eye_history[-WINDOW:]

        motion_ok = np.mean(motion_history) > MOTION_THRESHOLD
        eye_ok = len(set(eye_history)) > 1
        static_frames = sum(m < 1.5 for m in motion_history)

        liveness_raw = motion_ok and eye_ok and static_frames < STATIC_LIMIT

        if liveness_raw:
            liveness_until = now + LIVENESS_HOLD_TIME

        liveness_ok = now < liveness_until

        id_pred, conf = recognizer.predict(face_gray)
        nome = label_map.get(id_pred, "Desconhecido")
        if conf > 80:
            nome = "Desconhecido"

        if nome == "Desconhecido":
            cor = (0,0,255)
            texto = "Desconhecido"
        elif not liveness_ok:
            cor = (0,255,255)
            texto = f"{nome} - Identificado"
        else:
            cor = (0,255,0)
            texto = f"{nome} - Vivo - Acesso concedido"

        cv2.rectangle(frame, (x,y), (x+w,y+h), cor, 2)
        cv2.putText(frame, texto, (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, cor, 2)

    cv2.imshow("Reconhecimento Facial - Liveness Estável", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
